<a href="https://colab.research.google.com/github/CooL-Legend/CSCI544-DLM/blob/adi/notebooks/gsm8k/Experiment_0_Baseline_Test_Evaluation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open in Colab"/></a>


# Experiment 0: Baseline Test — Evaluation

Search-free LLaDA Confidence baseline (256 NFEs, $T=0$) on GSM8K, with optional LLaDA Entropy baseline.

Default: **20 training questions (prefix slicing with NUM_SAMPLES supported)** from the frozen shared split (`configs/gsm8k/experiment_samples.json`).

Every condition uses standard search-free LLaDA completion at $T=0$ over 256 output positions with the shared two-shot decomposition prompt (`gsm8k_shared_v2`). All search conditions (MEDAL MCTS) have been unified into Experiment 1.

Run top-to-bottom on an L4/A100 Colab GPU. Artifacts and a frozen manifest persist on Drive; preview cells read the exact saved run.


## 1. Verify GPU

In [1]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError("Select an L4 or A100 GPU runtime, then reconnect.")
gpu = torch.cuda.get_device_properties(0)
print(f"GPU: {gpu.name} | GPU memory: {gpu.total_memory / 2**30:.1f} GiB")
if gpu.total_memory < 20 * 2**30:
    raise RuntimeError("This notebook uses unquantized BF16 weights. Choose an L4/A100 runtime with more GPU memory; a 16 GB T4 is too small for this setup.")
if not torch.cuda.is_bf16_supported():
    raise RuntimeError("This GPU does not support the configured BF16 inference.")


GPU: NVIDIA L4 | GPU memory: 22.0 GiB


## 2. Get the source
Use the private repository on `adi` or upload the source bundle. Existing checkout changes are never discarded.

In [2]:
SOURCE_MODE = "github"  # Change to "upload" to use the delivered source ZIP.

from pathlib import Path
import getpass
import os
import subprocess
import sys
import tempfile

REPOSITORY = "CooL-Legend/CSCI544-DLM"
BRANCH = "adi"
REPO_URL = f"https://github.com/{REPOSITORY}.git"
REPO_DIR = Path("/content/CSCI544-DLM")

def run_git(args, token=None):
    env = {**os.environ, "GIT_TERMINAL_PROMPT": "0"}
    with tempfile.TemporaryDirectory() as tmp:
        if token:
            askpass = Path(tmp) / "askpass.py"
            askpass.write_text(
                "#!/usr/bin/env python3\nimport os, sys\n"
                "print('x-access-token' if 'username' in sys.argv[1].lower() else os.environ['COLAB_GH_TOKEN'])\n"
            )
            askpass.chmod(0o700)
            env.update(GIT_ASKPASS=str(askpass), COLAB_GH_TOKEN=token)
        return subprocess.run(["git", "-c", "credential.helper=", *args], env=env, capture_output=True, text=True)

def remote_git(args):
    result = run_git(args)
    if result.returncode:
        from google.colab import userdata
        try:
            token = userdata.get("GITHUB_TOKEN")
        except Exception:
            token = None
        if not token:
            token = getpass.getpass("GitHub token with read access to this private repository: ")
        try:
            result = run_git(args, token)
        finally:
            del token
    if result.returncode:
        raise RuntimeError(result.stderr.strip() or "GitHub clone/update failed.")

def ensure_checkout():
    def local_git(*args):
        return subprocess.check_output(["git", "-C", str(REPO_DIR), *args], text=True).strip()

    if REPO_DIR.exists():
        if not (REPO_DIR / ".git").is_dir():
            raise RuntimeError(f"{REPO_DIR} exists but is not a Git checkout. Choose another REPO_DIR.")
        if local_git("remote", "get-url", "origin") != REPO_URL:
            raise RuntimeError("The existing checkout belongs to a different repository.")
        if local_git("status", "--porcelain"):
            raise RuntimeError("The checkout has local changes. Save them or choose another REPO_DIR before updating.")
        fetch_specs = local_git("config", "--get-all", "remote.origin.fetch").splitlines()
        branch_ref = f"refs/heads/{BRANCH}:refs/remotes/origin/{BRANCH}"
        if not any(spec.lstrip("+") in (branch_ref, "refs/heads/*:refs/remotes/origin/*") for spec in fetch_specs):
            local_git("remote", "set-branches", "--add", "origin", BRANCH)
        remote_git(["-C", str(REPO_DIR), "fetch", "origin", branch_ref])
        exists = subprocess.run(["git", "-C", str(REPO_DIR), "show-ref", "--verify", "--quiet", f"refs/heads/{BRANCH}"]).returncode == 0
        if exists:
            local_git("switch", BRANCH)
        else:
            local_git("switch", "--create", BRANCH, "--track", f"origin/{BRANCH}")
        local_git("merge", "--ff-only", f"origin/{BRANCH}")
    else:
        remote_git(["clone", "--branch", BRANCH, "--single-branch", REPO_URL, str(REPO_DIR)])
    if local_git("branch", "--show-current") != BRANCH or local_git("rev-parse", "HEAD") != local_git("rev-parse", f"origin/{BRANCH}"):
        raise RuntimeError("The checkout differs from origin/adi. Review local commits before running experiments.")
    print(f"Code ready: {REPO_DIR} | branch: {BRANCH} | commit: {local_git('rev-parse', '--short', 'HEAD')}")

if SOURCE_MODE == "github":
    ensure_checkout()
elif SOURCE_MODE == "upload":
    # The delivered source ZIP works before these changes have been pushed to adi.
    from google.colab import files
    import io
    import uuid
    import zipfile
    uploaded = files.upload()
    archives = [name for name in uploaded if name.endswith(".zip")]
    if len(archives) != 1:
        raise RuntimeError("Upload exactly one experiment_0_colab_source.zip archive.")
    REPO_DIR = Path("/content/CSCI544-DLM-e0") / uuid.uuid4().hex[:8]
    REPO_DIR.mkdir(parents=True)
    with zipfile.ZipFile(io.BytesIO(uploaded[archives[0]])) as archive:
        for member in archive.infolist():
            target = (REPO_DIR / member.filename).resolve()
            if not target.is_relative_to(REPO_DIR.resolve()):
                raise RuntimeError("Unsafe source archive path.")
        archive.extractall(REPO_DIR)
    del uploaded
    print("Uploaded E0 source:", REPO_DIR)
else:
    raise ValueError("SOURCE_MODE must be github or upload.")
if not (REPO_DIR / "scripts/experiment_0_baseline_test.py").is_file():
    raise RuntimeError("E0 is not on this checkout yet. Set SOURCE_MODE='upload' and use the delivered source ZIP, or publish the E0 commit to adi.")


Code ready: /content/CSCI544-DLM | branch: adi | commit: f3d9664


## 3. Install LLaDA dependencies

In [3]:
subprocess.run([
    sys.executable, "-m", "pip", "install", "-q",
    "-r", str(REPO_DIR / "requirements/llada-colab.txt"),
], check=True)

# Install our package without replacing Colab's CUDA-enabled PyTorch.
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "-e", str(REPO_DIR)], check=True)
# Refresh editable-package paths in this already-running notebook kernel.
import site
for package_dir in [*site.getsitepackages(), site.getusersitepackages()]:
    site.addsitedir(package_dir)


## 4. Mount Drive and select cache locations

In [5]:
from google.colab import drive
import importlib

drive.mount("/content/drive")
DRIVE_ROOT = Path("/content/drive/MyDrive/CSCI544-DLM")
MODEL_FAMILY = "llada"
from dlm_project.models import profiles as model_profiles
from dlm_project.experiments import paths as experiment_paths
importlib.reload(model_profiles)
importlib.reload(experiment_paths)
profile = model_profiles.get_profile(MODEL_FAMILY)
MODEL_NAME = profile["repo_id"].split("/")[-1]
DRIVE_MODEL_DIR = DRIVE_ROOT / "models" / MODEL_NAME
RUNTIME_MODEL_DIR = Path("/content/models") / MODEL_NAME
COPY_MODEL_TO_RUNTIME = True
EXPERIMENT_TIMEZONE = "America/Los_Angeles"

DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
os.environ["HF_HUB_CACHE"] = str(DRIVE_ROOT / "cache/hub")
os.environ["HF_DATASETS_CACHE"] = str(DRIVE_ROOT / "cache/datasets")
MODEL_PATH = RUNTIME_MODEL_DIR if COPY_MODEL_TO_RUNTIME else DRIVE_MODEL_DIR
print("Persistent model:", DRIVE_MODEL_DIR)
print("Experiment root:", DRIVE_ROOT / profile["label"] / "GSM8K")


Mounted at /content/drive
Persistent model: /content/drive/MyDrive/CSCI544-DLM/models/LLaDA-8B-Instruct
Experiment root: /content/drive/MyDrive/CSCI544-DLM/LLaDA/GSM8K


## 5. Experiment controls
Search-free LLaDA baseline configuration. Search-free decoding uses greedy confidence at T=0 across 256 steps. Optional greedy entropy can be enabled via `INCLUDE_ENTROPY = True`.


In [8]:
import yaml

STAGE = "evaluation"
NUM_SAMPLES = None  # None uses all questions for the stage (dev: 5, eval: 20, test: 200). Or set an integer prefix.
INCLUDE_ENTROPY = False  # Set to True to include the optional search-free LLaDA Entropy baseline.
RUN_ORDER = "condition"

cfg = yaml.safe_load((REPO_DIR / "configs/gsm8k/experiment_0.yaml").read_text())
cfg.setdefault("model", {}).update(model_path=str(MODEL_PATH), device="cuda", local_files_only=True)
cfg.setdefault("data", {}).update(cache_dir=os.environ["HF_DATASETS_CACHE"])
if NUM_SAMPLES is not None:
    cfg["data"]["num_samples"] = NUM_SAMPLES
cfg.setdefault("evaluation", {}).update(stage=STAGE, run_order=RUN_ORDER)
if INCLUDE_ENTROPY:
    cfg["evaluation"]["methods"] = ["confidence", "entropy"]
cfg["experiment_root"] = str(DRIVE_ROOT)
cfg["experiment_timezone"] = EXPERIMENT_TIMEZONE
E0_MANIFEST = REPO_DIR / "configs/gsm8k/experiment_samples.json"
E0_CONFIG = Path("/content") / f"experiment-0-{STAGE}.yaml"
E0_CONFIG.write_text(yaml.safe_dump(cfg, sort_keys=False))
print(yaml.safe_dump(cfg, sort_keys=False))
print("Model revision:", profile["revision"])
print("Frozen samples manifest:", E0_MANIFEST)


common_config: configs/gsm8k/experiments_common.yaml
evaluation:
  stage: evaluation
  run_order: condition
  methods:
  - confidence
experiment_root: /content/drive/MyDrive/CSCI544-DLM
experiment_timezone: America/Los_Angeles
model:
  model_path: /content/models/LLaDA-8B-Instruct
  device: cuda
  local_files_only: true
data:
  cache_dir: /content/drive/MyDrive/CSCI544-DLM/cache/datasets

Model revision: 08b83a6feb34df1a6011b80c3c00c7563e963b07
Frozen samples manifest: /content/CSCI544-DLM/configs/gsm8k/experiment_samples.json


## 6. Cache the pinned model

In [9]:
command = [sys.executable, "-u", str(REPO_DIR / "scripts/cache_model.py"), "--model", MODEL_FAMILY, "--drive-dir", str(DRIVE_MODEL_DIR)]
if COPY_MODEL_TO_RUNTIME:
    command += ["--runtime-dir", str(RUNTIME_MODEL_DIR)]
subprocess.run(command, check=True)


CompletedProcess(args=['/usr/bin/python3', '-u', '/content/CSCI544-DLM/scripts/cache_model.py', '--model', 'llada', '--drive-dir', '/content/drive/MyDrive/CSCI544-DLM/models/LLaDA-8B-Instruct', '--runtime-dir', '/content/models/LLaDA-8B-Instruct'], returncode=0)

## 7. Run search-free baseline
Execute search-free LLaDA confidence (and optional entropy) decoding at T=0. Output streams per completed condition.


In [10]:
import json
import subprocess
import sys
import uuid
from pathlib import Path

E0_RUN_DIR = Path(f"/content/drive/MyDrive/CSCI-DLM/Llada/GSM8K/experiment_0/{STAGE}")
LAST_PARTIAL_RUN_DIR = None

def run_e0():
    global LAST_PARTIAL_RUN_DIR
    LAST_PARTIAL_RUN_DIR = None
    receipt = Path("/content") / f"e0-{STAGE}-{uuid.uuid4().hex}.json"
    command = [
        sys.executable, "-u", str(REPO_DIR / "scripts/experiment_0_baseline_test.py"),
        "--config", str(E0_CONFIG),
        "--stage", STAGE,
        "--run-order", RUN_ORDER,
        "--manifest", str(E0_MANIFEST),
        "--run-manifest", str(receipt),
    ]
    if NUM_SAMPLES is not None:
        command.extend(["--num-samples", str(NUM_SAMPLES)])
    if INCLUDE_ENTROPY:
        command.append("--include-entropy")

    process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                               text=True, bufsize=1, cwd=REPO_DIR)
    try:
        for line in process.stdout:
            print(line, end="", flush=True)
        returncode = process.wait()
    except KeyboardInterrupt:
        process.terminate()
        try:
            process.wait(timeout=10)
        except subprocess.TimeoutExpired:
            process.kill()
            process.wait()
        if receipt.is_file():
            LAST_PARTIAL_RUN_DIR = Path(json.loads(receipt.read_text())["run_dir"])
            print("Partial artifacts:", LAST_PARTIAL_RUN_DIR)
        raise
    if receipt.is_file():
        saved = json.loads(receipt.read_text())
        LAST_PARTIAL_RUN_DIR = Path(saved["run_dir"])
    if returncode:
        print("Partial artifacts:", LAST_PARTIAL_RUN_DIR)
        raise subprocess.CalledProcessError(returncode, command)
    if not receipt.is_file():
        raise RuntimeError("The E0 process produced no run receipt.")
    expected_conditions = ["confidence_t0", "entropy_t0"] if INCLUDE_ENTROPY else ["confidence_t0"]
    if (saved["status"] != "completed" or saved["stage"] != STAGE
            or saved.get("run_order") != RUN_ORDER
            or saved.get("condition_ids") != expected_conditions
            or saved["protocol_version"] != "gsm8k_shared_v2"):
        raise RuntimeError(f"E0 receipt does not describe this completed invocation: {saved}")
    return Path(saved["run_dir"])

E0_RUN_DIR = run_e0()


E0 artifacts: /content/drive/MyDrive/CSCI544-DLM/LLaDA/GSM8K/experiment_0/evaluation/fixed-matrix-v6/26:10:26:00:55:11
Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.

Loading checkpoint shards: 100%|██████████| 6/6 [00:29<00:00,  4.84s/it]
E0 execution order: condition confidence_t0
Starting confidence_t0 on 20 questions
I0000 00:00:1793001415.522491    5195 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1793001418.678891    5195 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1793001418.679415    5195 cudart

RuntimeError: E0 receipt does not describe this completed invocation: {'run_dir': '/content/drive/MyDrive/CSCI544-DLM/LLaDA/GSM8K/experiment_0/evaluation/fixed-matrix-v6/26:10:26:00:55:11', 'status': 'completed', 'protocol_version': 'e0_fixed_budget_v6', 'common_protocol_hash': '8b8c8019f06ad95e2c64ec0683e32d0c0814314d4485421c4b2955f6a7d8e09b', 'stage': 'evaluation', 'search_mode': 'none', 'run_order': 'condition', 'condition_ids': ['confidence_t0'], 'model_family': 'llada'}

## 8. Accuracy, compute, runtime and search depth
Paired metrics use questions with a terminal outcome under every condition. Partial runs retain observed counts and unfinished-forward costs. Report rebuilding performs no inference.
If stopped before all methods finish, `observed_summary.csv` retains each method’s metrics on its completed questions; the paired table waits for outcomes under every condition.

In [ ]:
from IPython.display import display, Markdown, Image, FileLink

REPORT_DIR = E0_RUN_DIR or LAST_PARTIAL_RUN_DIR
if REPORT_DIR is None:
    raise RuntimeError("Run E0 first, or set REPORT_DIR to one saved E0 run directory.")
# Rebuild from saved records only. This subprocess does not load weights or data.
subprocess.run([sys.executable, str(REPO_DIR / "scripts/experiment_0_baseline_test.py"),
                "--report-only", str(REPORT_DIR)], check=True, cwd=REPO_DIR)
display(Markdown((REPORT_DIR / "report/report.md").read_text()))
for name in ["accuracy_vs_total_nfes", "accuracy_vs_runtime"]:
    plot_file = REPORT_DIR / f"report/plots/{name}.png"
    if plot_file.is_file():
        display(Image(filename=str(plot_file)))
for name in ["summary.csv", "observed_summary.csv", "accuracy_gains.csv", "results.csv"]:
    link_file = REPORT_DIR / "report" / name
    if link_file.is_file():
        display(FileLink(str(link_file)))
print("Setup and separate overhead timings:")
print(json.dumps(json.loads((REPORT_DIR / "setup.json").read_text()), indent=2))


## 9. Inspect saved answers and termination boundaries

In [ ]:
from IPython.display import Code

QUESTION_INDEX = 0
records = [json.loads(line) for line in (REPORT_DIR / "predictions.jsonl").read_text().splitlines() if line.strip()]
example_ids = list(dict.fromkeys(row["example_id"] for row in records))
if not 0 <= QUESTION_INDEX < len(example_ids):
    raise ValueError(f"Choose QUESTION_INDEX in 0..{len(example_ids)-1}.")
selected = [row for row in records if row["example_id"] == example_ids[QUESTION_INDEX]]
print(selected[0]["example_id"], selected[0]["question"])
for row in selected:
    print("\n", row["condition"], "correct=", row["exact_match"],
          "predicted=", row["predicted_answer"], "expected=", row["reference_answer"])
    print("completion/total NFEs:", row["completion_nfes"], row["total_nfes"],
          "first EOS/EOT:", row["first_terminator_position"])
    print("Scored prefix:")
    display(Code(row["scoring_text"], language="text"))
    print("Full fixed-length output, including special tokens:")
    display(Code(row["raw_output"], language="text"))


In [ ]:
TRACE_CONDITION = "confidence_t0"
selected_condition = next(row for row in selected if row["condition"] == TRACE_CONDITION)
events = [json.loads(line) for line in (REPORT_DIR / selected_condition["trace_file"]).read_text().splitlines()]
print(f"Sampling trace for {TRACE_CONDITION} ({len(events)} steps recorded):")
if events:
    print(json.dumps(events[:3], indent=2))
